In [1]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=./infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

foundry_endpoint = ! terraform -chdir=./infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=./infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=./infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

sessionpool_management_endpoint_python = ! terraform -chdir=./infra output -raw sessionpool_management_endpoint_python
sessionpool_management_endpoint_python = sessionpool_management_endpoint_python.n
print("Session Pool Management Endpoint:", sessionpool_management_endpoint_python)

sessionpool_mcp_endpoint_python = ! terraform -chdir=./infra output -raw sessionpool_mcp_endpoint_python
sessionpool_mcp_endpoint_python = sessionpool_mcp_endpoint_python.n
print("MCP Session Pool Endpoint:", sessionpool_mcp_endpoint_python)

sessionpool_management_endpoint_shell = ! terraform -chdir=./infra output -raw sessionpool_management_endpoint_shell
sessionpool_management_endpoint_shell = sessionpool_management_endpoint_shell.n
print("Session Pool Management Endpoint (Shell):", sessionpool_management_endpoint_shell)

sessionpool_mcp_endpoint_shell = ! terraform -chdir=./infra output -raw sessionpool_mcp_endpoint_shell
sessionpool_mcp_endpoint_shell = sessionpool_mcp_endpoint_shell.n
print("MCP Session Pool Endpoint (Shell):", sessionpool_mcp_endpoint_shell)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4
Session Pool Management Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/sessionPools/acasessionpool-python
MCP Session Pool Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/sessionPools/acasessionpool-python/mcp
Session Pool Management Endpoint (Shell): https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/sessionPools/acasessionpool-shell
MCP Session Pool Endpoint (Shell): https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/s

In [1]:
%pip install langchain langchain-openai langchain-mcp-adapters langchain-azure-dynamic-sessions

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# model = ChatOpenAI(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     api_key="EMPTY",
#     model="google/gemma-4-31B-it",
#     streaming=True,
#     max_completion_tokens= 512
# )

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    max_completion_tokens=512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- helping with coding and problem-solving

I don’t have feelings, personal experiences, or a life story the way a person does, but I can carry on a conversation and adapt to what you need.

A few useful things to know about me:
- I generate responses based on patterns in data I was trained on.
- I can make mistakes, so it’s good to verify important information.
- I don’t automatically know personal details about you unless you share them in the conversation.
- My “knowledge” isn’t the same as live awareness of the world unless I’m given current info or tools.

If you want, I can also tell you about:
- how I work
- what I’m good at
- my limitations
- how to get the best results from me

In [4]:
from langchain.agents import create_agent
from langchain_azure_dynamic_sessions.tools import SessionsPythonREPLTool
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

def access_token_provider():
    token = credential.get_token("https://dynamicsessions.io/.default")
    return token.token

# get the management endpoint from the session pool in the Azure portal
toolPythonSession = SessionsPythonREPLTool(
    pool_management_endpoint=sessionpool_management_endpoint_python,
    access_token_provider=access_token_provider,
)

agent = create_agent(model=model, tools=[toolPythonSession])

async for step in agent.astream(
    {"messages": [{"role": "user", "content": "What is the current time in Tunisia and France ?"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the current time in Tunisia and France ?
================================== Ai Message ==================================

I don’t have live clock access in this chat, but I can tell you the time zones:

- **Tunisia**: usually **UTC+1** year-round
- **France**:
  - **UTC+1** in standard time
  - **UTC+2** during daylight saving time

So:

- In **winter**, **Tunisia and France are usually the same time**
- In **summer**, **France is usually 1 hour ahead of Tunisia**

If you want, I can also help you figure out the **current exact time** if you tell me your local time or timezone.


In [5]:
import io
import json

data = {"important_data": [1, 10, -1541]}
binary_io = io.BytesIO(json.dumps(data).encode("ascii"))

upload_metadata = toolPythonSession.upload_file(
    data=binary_io, remote_file_path="important_data.json"
)

code = f"""
import json

with open("{upload_metadata.full_path}") as f:
    data = json.load(f)

sum(data['important_data'])
"""

toolPythonSession.execute(code)

{'$id': '2',
 'status': 'Success',
 'stdout': '',
 'stderr': '',
 'result': -1530,
 'executionTimeInMilliseconds': 51}

In [ ]:
# from langchain_azure_dynamic_sessions import SessionsBashTool

# tool = SessionsBashTool(pool_management_endpoint=sessionpool_management_endpoint, access_token_provider=access_token_provider)

# response = tool.execute("echo Hello from the session!")

# # response

# # # upload a file to the session
# # tool.upload_file(local_file_path="./rg.tf", remote_file_path="/mnt/user/rg.tf")

# # # list files in the session
# # files = tool.list_files()

# # # download a file from the session
# # tool.download_file(remote_file_path="/mnt/user/output.txt", local_file_path="./output.txt")

In [14]:
from langchain_azure_dynamic_sessions.tools import SessionsBashTool

# get the management endpoint from the session pool in the Azure portal
toolBashSession = SessionsBashTool(
    pool_management_endpoint=sessionpool_management_endpoint_shell,
    access_token_provider=access_token_provider,
)

response = toolBashSession.execute("echo Hello from the session!")

print(json.dumps(response, indent=2))

{
  "identifier": "f570d912-bb2a-434b-bd71-36734823dcc7",
  "status": "0",
  "result": {
    "stdout": "Hello from the session!\n",
    "stderr": "",
    "executionTimeInMilliseconds": 2
  }
}


In [15]:
response = toolBashSession.execute("printenv")

print(json.dumps(response, indent=2))

{
  "identifier": "f570d912-bb2a-434b-bd71-36734823dcc7",
  "status": "0",
  "result": {
    "stdout": "PWD=/mnt/data\nENABLE_EGRESS=false\n_=/usr/bin/printenv\nHOME=/root\nCODE_INTERPRETER_CONFIG_FILE_PATH=/app/codeInterpreterConf.yaml\nSHLVL=1\nPATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin\n",
    "stderr": "",
    "executionTimeInMilliseconds": 3
  }
}


In [16]:
response = toolBashSession.execute("cat /app/codeInterpreterConf.yaml")

print(json.dumps(response, indent=2))

{
  "identifier": "f570d912-bb2a-434b-bd71-36734823dcc7",
  "status": "0",
  "result": {
    "stdout": "kind: CodeInterpreterConfig\napiVersion: v1\nmetadata:\n  name: code-interpreter\nspec:\n  logging:\n    logLevel: 1 # 0: debug, 1: info, 2: warn, 3: error\n    useISO8601DateFormat: false\n  kernel:\n    enabled: false\n    isFirstParty: true\n    port: 6003\n\n  healthProbe: # It will respond back /health endpoint with 200 status code if the container is healthy\n    initialDelaySeconds: 0\n\n  fileOperation:\n    maxUploadFileSizeLimitInMBs: 250\n    enableFullPathAccess: true\n",
    "stderr": "",
    "executionTimeInMilliseconds": 2
  }
}


In [20]:
response = toolBashSession.execute("curl ifconfig.me/ip")

print(json.dumps(response, indent=2))

{
  "identifier": "f570d912-bb2a-434b-bd71-36734823dcc7",
  "status": "0",
  "result": {
    "stdout": "4.165.18.176",
    "stderr": "  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100    12  100    12    0     0     27      0 --:--:-- --:--:-- --:--:--    27\r100    12  100    12    0     0     27      0 --:--:-- --:--:-- --:--:--    27\n",
    "executionTimeInMilliseconds": 447
  }
}


In [21]:
agent = create_agent(model=model, tools=[toolBashSession])

async for step in agent.astream(
    {"messages": [{"role": "user", "content": "Create a hello world flask app in a new remote environment. Then send a request to the app to show that it works."}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Create a hello world flask app in a new remote environment. Then send a request to the app to show that it works.
================================== Ai Message ==================================
Tool Calls:
  Bash (call_hGjjUqza3LKAsh8Ja9paDyss)
 Call ID: call_hGjjUqza3LKAsh8Ja9paDyss
  Args:
    bash_command: set -e
mkdir -p /tmp/flask_hello && cd /tmp/flask_hello
python3 -m venv venv
. venv/bin/activate
pip install --quiet flask
cat > app.py <<'PY'
from flask import Flask
app = Flask(__name__)

@app.route('/')
def hello():
    return 'Hello, World!'

if __name__ == '__main__':
    app.run(host='127.0.0.1', port=5000)
PY
nohup venv/bin/python app.py >/tmp/flask_hello/flask.log 2>&1 &
sleep 2
curl -s http://127.0.0.1:5000/
================================= Tool Message =================================
Name: Bash

{
  "stdout": "The virtual environment was not created successfully because ensurepip is not